Using gpu

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# =========================
# 1. LOAD MODEL
# =========================

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load base model (CPU safe)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

# Load your fine-tuned LoRA
model = PeftModel.from_pretrained(base_model, "v1-reply-model")

model.eval()

print("✅ Model loaded successfully")

✅ Model loaded successfully


In [8]:
# =========================
# 2. BUILD PROMPT
# =========================

def build_prompt(conversation):
    return f"""### Instruction:
Generate exactly 3 short, natural reply suggestions.

### Rules:
- Each reply must be different
- Keep replies short (1 sentence)
- Format strictly:
Reply 1: ...
Reply 2: ...
Reply 3: ...

### Conversation:
{conversation}

### Response:
"""


In [9]:

# =========================
# 3. GENERATE REPLIES
# =========================

def generate_replies(conversation):
    prompt = build_prompt(conversation)

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,        # 🔥 more space
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            repetition_penalty=1.2     # 🔥 reduce repetition
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # =========================
    # 4. CLEAN OUTPUT
    # =========================

    result = text.split("### Response:")[-1].strip()

    lines = [line.strip() for line in result.split("\n") if line.strip()]

    # Keep only reply lines
    replies = []
    for line in lines:
        if "Reply" in line:
            replies.append(line)

    # Ensure 3 replies
    while len(replies) < 3:
        replies.append(f"Reply {len(replies)+1}: ...")

    return replies[:3]


In [10]:

# =========================
# 5. TEST
# =========================

if __name__ == "__main__":
    conversation = """User: I'm tired
Assistant: Long day?
User: Yeah"""

    replies = generate_replies(conversation)

    print("\n🔥 GENERATED REPLIES:\n")
    for r in replies:
        print(r)


🔥 GENERATED REPLIES:

Reply 1: ...
Reply 2: ...
Reply 3: ...
